# Solace Browser Deployment Notebook 05 — iOS App Store

**Version:** 1.0.0 | **Auth:** 65537 | **Rung Target:** 65537

NORTHSTAR: archive a signed IPA via xcodebuild, upload to App Store Connect, submit to TestFlight, then submit for App Review with full SHA-256 evidence chain.

## Prerequisites

- macOS with Xcode installed (`xcode-select -p` returns valid path)
- Apple Developer Team ID in `$APPLE_TEAM_ID`
- Distribution signing identity installed in Keychain
- Provisioning profile at `$SOLACE_PROVISIONING_PROFILE` or auto-managed
- App Store Connect API key at `$APP_STORE_CONNECT_API_KEY_PATH` (AuthKey_*.p8)
- `$APP_STORE_CONNECT_KEY_ID` and `$APP_STORE_CONNECT_ISSUER_ID` set
- `VERSION` file in project root

In [ ]:
"""PREFLIGHT — verify Xcode project, signing profile, and Info.plist."""
import os
import subprocess
import json
import hashlib
import plistlib
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path('/home/phuc/projects/solace-browser')
IOS_DIR = PROJECT_ROOT / 'ios'
VERSION = (PROJECT_ROOT / 'VERSION').read_text(encoding='utf-8').strip()
TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
EVIDENCE_DIR = PROJECT_ROOT / 'scratch' / 'app-store-deploy' / TIMESTAMP
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR = EVIDENCE_DIR / 'archive'
EXPORT_DIR = EVIDENCE_DIR / 'export'

BUNDLE_ID = 'com.solaceagi.browser'
SCHEME = 'SolaceBrowser'

# --- Verify Xcode ---
xcode_path_result = subprocess.run(
    ['xcode-select', '-p'],
    capture_output=True, text=True, timeout=10
)
assert xcode_path_result.returncode == 0, \
    f'xcode-select failed: {xcode_path_result.stderr}'
XCODE_PATH = xcode_path_result.stdout.strip()
print(f'Xcode path: {XCODE_PATH}')

xcode_version_result = subprocess.run(
    ['xcodebuild', '-version'],
    capture_output=True, text=True, timeout=10
)
assert xcode_version_result.returncode == 0, \
    f'xcodebuild -version failed: {xcode_version_result.stderr}'
XCODE_VERSION = xcode_version_result.stdout.strip()
print(f'Xcode version: {XCODE_VERSION}')

# --- Verify Xcode project or workspace ---
XCWORKSPACE = IOS_DIR / f'{SCHEME}.xcworkspace'
XCODEPROJ = IOS_DIR / f'{SCHEME}.xcodeproj'
if XCWORKSPACE.is_dir():
    BUILD_TARGET = ['-workspace', str(XCWORKSPACE)]
    print(f'Using workspace: {XCWORKSPACE.name}')
elif XCODEPROJ.is_dir():
    BUILD_TARGET = ['-project', str(XCODEPROJ)]
    print(f'Using project: {XCODEPROJ.name}')
else:
    raise FileNotFoundError(
        f'Neither {XCWORKSPACE} nor {XCODEPROJ} found'
    )

# --- Verify Info.plist ---
INFO_PLIST = IOS_DIR / f'{SCHEME}' / 'Info.plist'
assert INFO_PLIST.is_file(), f'Info.plist not found at {INFO_PLIST}'
with INFO_PLIST.open('rb') as f:
    plist_data = plistlib.load(f)

plist_bundle_id = plist_data.get('CFBundleIdentifier', '')
plist_version = plist_data.get('CFBundleShortVersionString', '')
plist_build = plist_data.get('CFBundleVersion', '')
# Bundle ID may use $(PRODUCT_BUNDLE_IDENTIFIER) variable
if '$(' not in plist_bundle_id:
    assert plist_bundle_id == BUNDLE_ID, \
        f'Info.plist bundle ID mismatch: {plist_bundle_id} != {BUNDLE_ID}'
print(f'Info.plist: version={plist_version}, build={plist_build}')

# --- Verify Apple Developer Team ID ---
APPLE_TEAM_ID = os.environ.get('APPLE_TEAM_ID', '')
assert APPLE_TEAM_ID, 'APPLE_TEAM_ID env var not set'
print(f'Team ID: {APPLE_TEAM_ID}')

# --- Verify signing identity ---
identity_result = subprocess.run(
    ['security', 'find-identity', '-v', '-p', 'codesigning'],
    capture_output=True, text=True, timeout=15
)
assert 'Apple Distribution' in identity_result.stdout or \
       'iPhone Distribution' in identity_result.stdout, \
    'No Apple Distribution signing identity found in Keychain'
print('Distribution signing identity: found')

# --- Verify App Store Connect API key ---
ASC_KEY_PATH = os.environ.get('APP_STORE_CONNECT_API_KEY_PATH', '')
ASC_KEY_ID = os.environ.get('APP_STORE_CONNECT_KEY_ID', '')
ASC_ISSUER_ID = os.environ.get('APP_STORE_CONNECT_ISSUER_ID', '')
assert ASC_KEY_PATH and Path(ASC_KEY_PATH).is_file(), \
    f'APP_STORE_CONNECT_API_KEY_PATH not set or missing: {ASC_KEY_PATH!r}'
assert ASC_KEY_ID, 'APP_STORE_CONNECT_KEY_ID env var not set'
assert ASC_ISSUER_ID, 'APP_STORE_CONNECT_ISSUER_ID env var not set'
print(f'ASC API Key ID: {ASC_KEY_ID}')

preflight = {
    'status': 'ok',
    'timestamp': TIMESTAMP,
    'version': VERSION,
    'plist_version': plist_version,
    'plist_build': plist_build,
    'bundle_id': BUNDLE_ID,
    'scheme': SCHEME,
    'xcode_version': XCODE_VERSION,
    'team_id': APPLE_TEAM_ID,
    'asc_key_id': ASC_KEY_ID,
    'evidence_dir': str(EVIDENCE_DIR),
}
(EVIDENCE_DIR / 'preflight.json').write_text(
    json.dumps(preflight, indent=2), encoding='utf-8'
)
print(json.dumps(preflight, indent=2))

In [ ]:
"""BEFORE SNAPSHOT — check current TestFlight and App Store status via altool."""

# --- Query App Store Connect for current build status ---
# Use xcrun altool to list builds (requires API key auth)
asc_auth_args = [
    '--apiKey', ASC_KEY_ID,
    '--apiIssuer', ASC_ISSUER_ID,
]

# List current app info
list_result = subprocess.run(
    ['xcrun', 'altool', '--list-apps'] + asc_auth_args
    + ['--output-format', 'json'],
    capture_output=True, text=True, timeout=60
)

before_snapshot = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'bundle_id': BUNDLE_ID,
    'query_status': 'ok' if list_result.returncode == 0 else 'error',
}

if list_result.returncode == 0:
    try:
        apps_data = json.loads(list_result.stdout)
        before_snapshot['apps'] = apps_data
    except json.JSONDecodeError:
        before_snapshot['raw_output'] = list_result.stdout[:2000]
else:
    before_snapshot['error'] = list_result.stderr[:1000]
    print(f'WARNING: altool --list-apps returned non-zero (may be first upload)')

# --- Check TestFlight builds via App Store Connect API (REST) ---
import urllib.request
import time
import jwt  # PyJWT

def make_asc_token() -> str:
    """Create a short-lived JWT for App Store Connect API."""
    key_data = Path(ASC_KEY_PATH).read_bytes()
    now = int(time.time())
    payload = {
        'iss': ASC_ISSUER_ID,
        'iat': now,
        'exp': now + 1200,  # 20 minutes
        'aud': 'appstoreconnect-v1',
    }
    return jwt.encode(payload, key_data, algorithm='ES256',
                      headers={'kid': ASC_KEY_ID})

asc_token = make_asc_token()
asc_headers = {
    'Authorization': f'Bearer {asc_token}',
    'Content-Type': 'application/json',
}

# Fetch app by bundle ID
apps_url = f'https://api.appstoreconnect.apple.com/v1/apps?filter[bundleId]={BUNDLE_ID}'
req = urllib.request.Request(apps_url, headers=asc_headers)
try:
    resp = urllib.request.urlopen(req, timeout=30)
    apps_json = json.loads(resp.read())
    app_data = apps_json.get('data', [])
    if app_data:
        APP_ID = app_data[0]['id']
        before_snapshot['app_store_connect_app_id'] = APP_ID
        before_snapshot['app_name'] = app_data[0]['attributes'].get('name', '')
        print(f'App found: {before_snapshot["app_name"]} (ID: {APP_ID})')

        # Fetch latest builds
        builds_url = (f'https://api.appstoreconnect.apple.com/v1/builds'
                      f'?filter[app]={APP_ID}&sort=-uploadedDate&limit=5')
        builds_req = urllib.request.Request(builds_url, headers=asc_headers)
        builds_resp = urllib.request.urlopen(builds_req, timeout=30)
        builds_json = json.loads(builds_resp.read())
        before_snapshot['recent_builds'] = [
            {
                'version': b['attributes'].get('version', ''),
                'uploadedDate': b['attributes'].get('uploadedDate', ''),
                'processingState': b['attributes'].get('processingState', ''),
            }
            for b in builds_json.get('data', [])
        ]
    else:
        APP_ID = None
        print('App not yet registered in App Store Connect')
except urllib.error.HTTPError as e:
    before_snapshot['asc_api_error'] = f'HTTP {e.code}'
    APP_ID = None
    print(f'App Store Connect API error: {e.code}')

(EVIDENCE_DIR / 'before-snapshot.json').write_text(
    json.dumps(before_snapshot, indent=2), encoding='utf-8'
)
print(json.dumps(before_snapshot, indent=2))

In [ ]:
"""BUILD — xcodebuild archive and export IPA."""

ARCHIVE_PATH = ARCHIVE_DIR / f'{SCHEME}.xcarchive'

# --- Clean build ---
clean_result = subprocess.run(
    ['xcodebuild', 'clean'] + BUILD_TARGET
    + ['-scheme', SCHEME, '-configuration', 'Release'],
    cwd=str(IOS_DIR),
    capture_output=True, text=True, timeout=120
)
assert clean_result.returncode == 0, \
    f'xcodebuild clean failed: {clean_result.stderr[-500:]}'
print('xcodebuild clean: OK')

# --- Archive ---
archive_result = subprocess.run(
    ['xcodebuild', 'archive'] + BUILD_TARGET
    + ['-scheme', SCHEME,
       '-configuration', 'Release',
       '-archivePath', str(ARCHIVE_PATH),
       '-destination', 'generic/platform=iOS',
       f'DEVELOPMENT_TEAM={APPLE_TEAM_ID}',
       'CODE_SIGN_STYLE=Automatic'],
    cwd=str(IOS_DIR),
    capture_output=True, text=True, timeout=600
)

archive_log_path = EVIDENCE_DIR / 'xcodebuild-archive.log'
archive_log_path.write_text(
    f'STDOUT:\n{archive_result.stdout}\n\nSTDERR:\n{archive_result.stderr}',
    encoding='utf-8'
)
assert archive_result.returncode == 0, \
    f'xcodebuild archive failed (log: {archive_log_path}): {archive_result.stderr[-500:]}'
assert ARCHIVE_PATH.is_dir(), f'Archive not created at {ARCHIVE_PATH}'
print(f'Archive created: {ARCHIVE_PATH}')

# --- Create export options plist ---
export_options = {
    'method': 'app-store',
    'teamID': APPLE_TEAM_ID,
    'uploadSymbols': True,
    'compileBitcode': False,
    'destination': 'export',
    'signingStyle': 'automatic',
}
export_plist_path = EVIDENCE_DIR / 'ExportOptions.plist'
with export_plist_path.open('wb') as f:
    plistlib.dump(export_options, f)
print(f'Export options written: {export_plist_path}')

# --- Export IPA ---
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_result = subprocess.run(
    ['xcodebuild', '-exportArchive',
     '-archivePath', str(ARCHIVE_PATH),
     '-exportPath', str(EXPORT_DIR),
     '-exportOptionsPlist', str(export_plist_path)],
    capture_output=True, text=True, timeout=300
)

export_log_path = EVIDENCE_DIR / 'xcodebuild-export.log'
export_log_path.write_text(
    f'STDOUT:\n{export_result.stdout}\n\nSTDERR:\n{export_result.stderr}',
    encoding='utf-8'
)
assert export_result.returncode == 0, \
    f'xcodebuild export failed (log: {export_log_path}): {export_result.stderr[-500:]}'

# --- Locate IPA ---
ipa_candidates = list(EXPORT_DIR.glob('*.ipa'))
assert len(ipa_candidates) > 0, f'No IPA found in {EXPORT_DIR}'
IPA_PATH = ipa_candidates[0]

# --- Compute IPA hash ---
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

IPA_HASH = sha256_file(IPA_PATH)
IPA_SIZE = IPA_PATH.stat().st_size

build_evidence = {
    'status': 'ok',
    'archive_path': str(ARCHIVE_PATH),
    'ipa_path': str(IPA_PATH),
    'ipa_name': IPA_PATH.name,
    'ipa_sha256': IPA_HASH,
    'ipa_size_bytes': IPA_SIZE,
    'ipa_size_mb': round(IPA_SIZE / (1024 * 1024), 2),
    'version': VERSION,
    'build_number': plist_build,
}
(EVIDENCE_DIR / 'build-evidence.json').write_text(
    json.dumps(build_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(build_evidence, indent=2))

In [ ]:
"""UPLOAD — upload IPA to App Store Connect via xcrun altool or notarytool."""

# --- Upload via xcrun altool ---
upload_result = subprocess.run(
    ['xcrun', 'altool', '--upload-app',
     '--type', 'ios',
     '--file', str(IPA_PATH),
     '--apiKey', ASC_KEY_ID,
     '--apiIssuer', ASC_ISSUER_ID,
     '--output-format', 'json'],
    capture_output=True, text=True, timeout=600
)

upload_log_path = EVIDENCE_DIR / 'altool-upload.log'
upload_log_path.write_text(
    f'STDOUT:\n{upload_result.stdout}\n\nSTDERR:\n{upload_result.stderr}',
    encoding='utf-8'
)
assert upload_result.returncode == 0, \
    f'altool upload failed (log: {upload_log_path}): {upload_result.stderr[-500:]}'

# Parse upload response
try:
    upload_response = json.loads(upload_result.stdout)
except json.JSONDecodeError:
    upload_response = {'raw': upload_result.stdout[:2000]}

print(f'Upload completed successfully')

# --- Wait for App Store Connect processing ---
print('Waiting for App Store Connect to process the build...')
print('(This can take 5-30 minutes. Check App Store Connect dashboard.)')

MAX_WAIT_SECONDS = 1800  # 30 minutes
POLL_INTERVAL = 60  # 1 minute
elapsed = 0
processing_complete = False

while elapsed < MAX_WAIT_SECONDS:
    asc_token = make_asc_token()
    asc_headers = {
        'Authorization': f'Bearer {asc_token}',
        'Content-Type': 'application/json',
    }
    builds_url = (f'https://api.appstoreconnect.apple.com/v1/builds'
                  f'?filter[app]={APP_ID}'
                  f'&filter[version]={plist_build}'
                  f'&sort=-uploadedDate&limit=1')
    builds_req = urllib.request.Request(builds_url, headers=asc_headers)
    try:
        builds_resp = urllib.request.urlopen(builds_req, timeout=30)
        builds_json = json.loads(builds_resp.read())
        builds = builds_json.get('data', [])
        if builds:
            state = builds[0]['attributes'].get('processingState', '')
            print(f'  [{elapsed}s] Processing state: {state}')
            if state == 'VALID':
                processing_complete = True
                BUILD_ID = builds[0]['id']
                break
            elif state == 'INVALID':
                raise RuntimeError(
                    f'Build processing failed: INVALID. '
                    f'Check App Store Connect for details.'
                )
    except urllib.error.HTTPError as e:
        print(f'  [{elapsed}s] API poll error: HTTP {e.code}')
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL

assert processing_complete, \
    f'Build processing did not complete within {MAX_WAIT_SECONDS}s'

upload_evidence = {
    'status': 'ok',
    'upload_response': upload_response,
    'build_id': BUILD_ID,
    'processing_state': 'VALID',
    'processing_time_seconds': elapsed,
    'timestamp': datetime.now(timezone.utc).isoformat(),
}
(EVIDENCE_DIR / 'upload-evidence.json').write_text(
    json.dumps(upload_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(upload_evidence, indent=2))

In [ ]:
"""TESTFLIGHT — submit build to TestFlight for beta testing.

Adds the processed build to a beta group for internal/external testing.
"""

# --- Add build to beta group via App Store Connect API ---
# First, get or create a beta group
asc_token = make_asc_token()
asc_headers = {
    'Authorization': f'Bearer {asc_token}',
    'Content-Type': 'application/json',
}

BETA_GROUP_NAME = 'Solace Internal Testers'

# List existing beta groups
groups_url = (f'https://api.appstoreconnect.apple.com/v1/betaGroups'
              f'?filter[app]={APP_ID}')
groups_req = urllib.request.Request(groups_url, headers=asc_headers)
groups_resp = urllib.request.urlopen(groups_req, timeout=30)
groups_json = json.loads(groups_resp.read())

beta_group_id = None
for group in groups_json.get('data', []):
    if group['attributes']['name'] == BETA_GROUP_NAME:
        beta_group_id = group['id']
        break

if beta_group_id is None:
    # Create the beta group
    create_group_body = json.dumps({
        'data': {
            'type': 'betaGroups',
            'attributes': {
                'name': BETA_GROUP_NAME,
                'isInternalGroup': True,
            },
            'relationships': {
                'app': {
                    'data': {'type': 'apps', 'id': APP_ID}
                }
            }
        }
    }).encode('utf-8')
    create_req = urllib.request.Request(
        'https://api.appstoreconnect.apple.com/v1/betaGroups',
        data=create_group_body,
        headers=asc_headers,
        method='POST'
    )
    create_resp = urllib.request.urlopen(create_req, timeout=30)
    create_json = json.loads(create_resp.read())
    beta_group_id = create_json['data']['id']
    print(f'Created beta group: {BETA_GROUP_NAME} (ID: {beta_group_id})')
else:
    print(f'Found beta group: {BETA_GROUP_NAME} (ID: {beta_group_id})')

# --- Add build to beta group ---
add_build_body = json.dumps({
    'data': [{'type': 'builds', 'id': BUILD_ID}]
}).encode('utf-8')

add_build_url = (f'https://api.appstoreconnect.apple.com/v1'
                 f'/betaGroups/{beta_group_id}/relationships/builds')
add_build_req = urllib.request.Request(
    add_build_url,
    data=add_build_body,
    headers=asc_headers,
    method='POST'
)
try:
    urllib.request.urlopen(add_build_req, timeout=30)
    print(f'Build {BUILD_ID} added to TestFlight group: {BETA_GROUP_NAME}')
except urllib.error.HTTPError as e:
    if e.code == 409:
        print(f'Build {BUILD_ID} already in TestFlight group (409 conflict — OK)')
    else:
        raise

# --- Set beta app review info (what's new) ---
localization_body = json.dumps({
    'data': {
        'type': 'betaBuildLocalizations',
        'attributes': {
            'locale': 'en-US',
            'whatsNew': f'Solace Browser v{VERSION} beta build. '
                        f'See release notes for details.'
        },
        'relationships': {
            'build': {
                'data': {'type': 'builds', 'id': BUILD_ID}
            }
        }
    }
}).encode('utf-8')
loc_req = urllib.request.Request(
    'https://api.appstoreconnect.apple.com/v1/betaBuildLocalizations',
    data=localization_body,
    headers=asc_headers,
    method='POST'
)
try:
    urllib.request.urlopen(loc_req, timeout=30)
    print('Beta build localization (whatsNew) set')
except urllib.error.HTTPError as e:
    if e.code == 409:
        print('Beta localization already exists (409 — OK)')
    else:
        raise

testflight_evidence = {
    'status': 'ok',
    'build_id': BUILD_ID,
    'beta_group': BETA_GROUP_NAME,
    'beta_group_id': beta_group_id,
    'timestamp': datetime.now(timezone.utc).isoformat(),
}
(EVIDENCE_DIR / 'testflight-evidence.json').write_text(
    json.dumps(testflight_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(testflight_evidence, indent=2))

In [ ]:
"""SUBMIT — submit build for App Review.

IMPORTANT: This is a manual approval step. After submission, the build
enters Apple's review queue. Typical review time: 24-48 hours.
Do NOT proceed to Evidence Seal until review status is confirmed.
"""

# --- Confirm operator intent ---
SUBMIT_FOR_REVIEW = os.environ.get('SOLACE_SUBMIT_FOR_REVIEW', 'false')
if SUBMIT_FOR_REVIEW.lower() != 'true':
    print('SUBMIT_FOR_REVIEW is not set to true.')
    print('Set SOLACE_SUBMIT_FOR_REVIEW=true to proceed with App Review submission.')
    print('Skipping submission — TestFlight build is available for testing.')
    review_evidence = {
        'status': 'skipped',
        'reason': 'SOLACE_SUBMIT_FOR_REVIEW not set to true',
        'build_id': BUILD_ID,
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }
else:
    # --- Create App Store Version ---
    asc_token = make_asc_token()
    asc_headers = {
        'Authorization': f'Bearer {asc_token}',
        'Content-Type': 'application/json',
    }

    # Create new app store version
    version_body = json.dumps({
        'data': {
            'type': 'appStoreVersions',
            'attributes': {
                'platform': 'IOS',
                'versionString': VERSION,
                'releaseType': 'MANUAL',
            },
            'relationships': {
                'app': {
                    'data': {'type': 'apps', 'id': APP_ID}
                },
                'build': {
                    'data': {'type': 'builds', 'id': BUILD_ID}
                }
            }
        }
    }).encode('utf-8')

    version_req = urllib.request.Request(
        'https://api.appstoreconnect.apple.com/v1/appStoreVersions',
        data=version_body,
        headers=asc_headers,
        method='POST'
    )
    version_resp = urllib.request.urlopen(version_req, timeout=30)
    version_json = json.loads(version_resp.read())
    version_id = version_json['data']['id']
    print(f'Created App Store Version: {VERSION} (ID: {version_id})')

    # --- Submit for review ---
    submission_body = json.dumps({
        'data': {
            'type': 'appStoreVersionSubmissions',
            'relationships': {
                'appStoreVersion': {
                    'data': {'type': 'appStoreVersions', 'id': version_id}
                }
            }
        }
    }).encode('utf-8')

    submit_req = urllib.request.Request(
        'https://api.appstoreconnect.apple.com/v1/appStoreVersionSubmissions',
        data=submission_body,
        headers=asc_headers,
        method='POST'
    )
    submit_resp = urllib.request.urlopen(submit_req, timeout=30)
    submit_json = json.loads(submit_resp.read())
    print(f'Submitted for App Review: {json.dumps(submit_json, indent=2)}')

    review_evidence = {
        'status': 'submitted',
        'build_id': BUILD_ID,
        'version_id': version_id,
        'version_string': VERSION,
        'release_type': 'MANUAL',
        'note': 'Awaiting Apple review (24-48 hours typical)',
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }

(EVIDENCE_DIR / 'review-evidence.json').write_text(
    json.dumps(review_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(review_evidence, indent=2))

In [ ]:
"""EVIDENCE SEAL — SHA-256 report with version, build number, IPA hash.

Produces a tamper-evident evidence chain linking every artifact
from this deployment session.
"""

# --- Collect all evidence file hashes ---
evidence_files = sorted(EVIDENCE_DIR.glob('*.json'))
evidence_hashes = {}
for ef in evidence_files:
    evidence_hashes[ef.name] = sha256_file(ef)

# --- Build hash chain ---
chain_input = '\n'.join(
    f'{h}  {name}' for name, h in sorted(evidence_hashes.items())
)
chain_hash = hashlib.sha256(chain_input.encode('utf-8')).hexdigest()

# --- Git state ---
git_sha = subprocess.run(
    ['git', 'rev-parse', 'HEAD'],
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True, timeout=10
).stdout.strip()

git_dirty = subprocess.run(
    ['git', 'status', '--porcelain'],
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True, timeout=10
).stdout.strip()

seal = {
    'seal': 'SOLACE-IOS-DEPLOY',
    'auth': 65537,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'version': VERSION,
    'build_number': plist_build,
    'bundle_id': BUNDLE_ID,
    'scheme': SCHEME,
    'ipa_name': IPA_PATH.name,
    'ipa_sha256': IPA_HASH,
    'ipa_size_bytes': IPA_SIZE,
    'xcode_version': XCODE_VERSION,
    'team_id': APPLE_TEAM_ID,
    'git_sha': git_sha,
    'git_dirty': bool(git_dirty),
    'evidence_chain': {
        'files': evidence_hashes,
        'chain_hash': chain_hash,
    },
}

seal_path = EVIDENCE_DIR / 'evidence-seal.json'
seal_path.write_text(json.dumps(seal, indent=2), encoding='utf-8')

# --- Write human-readable report ---
report_lines = [
    f'# iOS App Store Deploy Evidence — v{VERSION}',
    f'',
    f'**Timestamp:** {seal["timestamp"]}',
    f'**Auth:** {seal["auth"]}',
    f'**Bundle ID:** {BUNDLE_ID}',
    f'**Version:** v{VERSION} (build {plist_build})',
    f'**Xcode:** {XCODE_VERSION}',
    f'**Team ID:** {APPLE_TEAM_ID}',
    f'**Git SHA:** {git_sha}',
    f'**Git Dirty:** {bool(git_dirty)}',
    f'',
    f'## IPA',
    f'- **File:** {IPA_PATH.name}',
    f'- **SHA-256:** `{IPA_HASH}`',
    f'- **Size:** {IPA_SIZE:,} bytes ({round(IPA_SIZE / (1024*1024), 2)} MB)',
    f'',
    f'## Evidence Chain',
    f'- **Chain Hash:** `{chain_hash}`',
    f'',
    f'### Evidence Files',
]
for name, h in sorted(evidence_hashes.items()):
    report_lines.append(f'- `{h}  {name}`')

report_path = EVIDENCE_DIR / 'deploy-report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

print(f'Evidence sealed: {seal_path}')
print(f'Report written: {report_path}')
print(f'Chain hash: {chain_hash}')
print(json.dumps(seal, indent=2))